# Module 1: Dependence Measures Comparison

## Learning Objectives

By completing this notebook, you will:
1. Compare Pearson, Spearman, distance correlation, and HSIC on the same dataset
2. Identify cases where each measure gives a meaningfully different feature ranking
3. Benchmark computational cost across measures at different sample sizes
4. Build a practical recommendation framework for choosing a dependence measure

## Prerequisites
- Guide 02: Advanced Dependence Measures
- Notebook 01: MI deep dive

## Estimated Time: 15 minutes

---

## 1. Setup and Implementations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import cdist
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Setup complete.")

In [ ]:
# -------------------------------------------------------------------------
# All dependence measures in one place
# -------------------------------------------------------------------------

def pearson_abs(x: np.ndarray, y: np.ndarray) -> float:
    """Absolute Pearson correlation. Symmetric, range [0,1]."""
    r, _ = pearsonr(x, y)
    return abs(r)


def spearman_abs(x: np.ndarray, y: np.ndarray) -> float:
    """Absolute Spearman rank correlation. Monotone-invariant."""
    r, _ = spearmanr(x, y)
    return abs(r)


def distance_correlation(x: np.ndarray, y: np.ndarray) -> float:
    """
    Distance correlation dCor(X, Y) in [0, 1].
    Detects arbitrary nonlinear dependencies; handles multivariate X, Y.
    """
    x2d = x.reshape(-1, 1) if x.ndim == 1 else x
    y2d = y.reshape(-1, 1) if y.ndim == 1 else y

    def double_center(D: np.ndarray) -> np.ndarray:
        row = D.mean(axis=1, keepdims=True)
        col = D.mean(axis=0, keepdims=True)
        return D - row - col + D.mean()

    A = double_center(cdist(x2d, x2d))
    B = double_center(cdist(y2d, y2d))

    dcov2_xy = (A * B).mean()
    dcov2_xx = (A * A).mean()
    dcov2_yy = (B * B).mean()

    denom = np.sqrt(dcov2_xx * dcov2_yy)
    return float(np.sqrt(max(0.0, dcov2_xy) / denom)) if denom > 1e-10 else 0.0


def hsic_rbf(x: np.ndarray, y: np.ndarray) -> float:
    """
    HSIC with RBF kernels. Bandwidth via median heuristic.
    Returns value in [0, inf) — higher means more dependence.
    """
    x2d = x.reshape(-1, 1) if x.ndim == 1 else x
    y2d = y.reshape(-1, 1) if y.ndim == 1 else y
    n = x2d.shape[0]

    def rbf(Z: np.ndarray) -> np.ndarray:
        D2 = cdist(Z, Z, metric='sqeuclidean')
        sigma = np.sqrt(0.5 * np.median(D2[D2 > 0]))
        return np.exp(-D2 / (2 * sigma**2 + 1e-10))

    K = rbf(x2d)
    L = rbf(y2d)
    H = np.eye(n) - np.ones((n, n)) / n
    return float(np.trace(K @ H @ L @ H) / (n - 1)**2)


def mi_sklearn_score(x: np.ndarray, y: np.ndarray) -> float:
    """sklearn adaptive MI (KSG-based). For classification targets."""
    return float(mutual_info_classif(
        x.reshape(-1, 1), y, n_neighbors=5, random_state=42)[0])


MEASURES = {
    'Pearson |r|':   pearson_abs,
    'Spearman |ρ|':  spearman_abs,
    'dCor':          distance_correlation,
    'HSIC (RBF)':    hsic_rbf,
    'MI (sklearn)':  mi_sklearn_score,
}

print("Dependence measures defined:", list(MEASURES.keys()))

## 2. Synthetic Benchmark: Known Dependency Types

We generate six relationship types with known ground truth and test which measure detects each one. All relationships have the same Pearson correlation by construction — the signal is in the higher-order structure.

In [ ]:
n = 600
X_uni = np.random.uniform(-2, 2, n)
eps   = lambda scale=0.3: np.random.normal(0, scale, n)

# Six dependency types — all with the target as a discrete binary label
dependency_types = {
    'Linear':          (X_uni + eps(),                       X_uni > 0),
    'Quadratic':       (X_uni**2 + eps(),                    X_uni > 0),
    'Step function':   (np.sign(X_uni) + eps(0.15),          X_uni > 0),
    'Sinusoidal':      (np.sin(3 * np.pi * X_uni) + eps(),   X_uni > 0),
    'Variance shift':  (X_uni * (1 + (X_uni > 0).astype(float) * 2) + eps(0.1),
                        X_uni > 0),
    'Independent':     (np.random.uniform(-2, 2, n),          X_uni > 0),
}

# Compute all measures on all dependency types
records = []
for dep_name, (feat_vals, target_vals) in dependency_types.items():
    row = {'Dependency': dep_name}
    target_int = target_vals.astype(int)

    for measure_name, fn in MEASURES.items():
        try:
            if measure_name == 'MI (sklearn)':
                score = fn(feat_vals, target_int)
            else:
                score = fn(feat_vals, target_vals.astype(float))
        except Exception:
            score = np.nan
        row[measure_name] = round(score, 4)
    records.append(row)

bench_df = pd.DataFrame(records).set_index('Dependency')
print("Dependence measure scores by relationship type:")
print(bench_df.to_string())
print("\nHigher = more detected dependence. Independent row should be near zero.")

In [ ]:
# Heatmap of detection scores (normalise per measure for visual comparison)
bench_norm = bench_df.copy()
for col in bench_norm.columns:
    col_max = bench_norm[col].max()
    if col_max > 0:
        bench_norm[col] = bench_norm[col] / col_max

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(bench_norm, annot=bench_df, fmt='.3f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Normalised score'})
ax.set_title('Dependence Detection by Measure and Relationship Type\n'
             '(scores normalised per column; cell values = raw scores)',
             fontsize=12)
ax.set_xlabel('Dependence measure')
ax.set_ylabel('Relationship type')
plt.tight_layout()
plt.savefig('dependence_detection_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Real Data: Feature Ranking Comparison

We apply all measures to the breast cancer dataset and compare how they rank the 30 features. Disagreements between measures reveal features with specific dependency types.

In [ ]:
# Load breast cancer dataset
data = load_breast_cancer()
X_bc = pd.DataFrame(data.data, columns=data.feature_names)
y_bc = data.target

scaler = StandardScaler()
X_bc_sc = pd.DataFrame(scaler.fit_transform(X_bc), columns=X_bc.columns)

print(f"Breast cancer: {X_bc.shape[0]} samples, {X_bc.shape[1]} features")
print("Computing all 5 measures × 30 features...")

# Compute scores for each feature × measure
# Note: HSIC is O(N^2) — takes a few seconds per feature
all_scores = {meas: {} for meas in MEASURES}

for feat in X_bc_sc.columns:
    x_vals = X_bc_sc[feat].values
    for meas_name, fn in MEASURES.items():
        if meas_name == 'MI (sklearn)':
            score = fn(x_vals, y_bc)
        else:
            score = fn(x_vals, y_bc.astype(float))
        all_scores[meas_name][feat] = score

scores_df = pd.DataFrame(all_scores)
print("Computation complete.")
print("\nTop 5 features by each measure:")
for col in scores_df.columns:
    top5 = scores_df[col].nlargest(5).index.tolist()
    print(f"  {col}: {top5}")

In [ ]:
# Rank features by each measure
rank_df = scores_df.rank(ascending=False).astype(int)

# Plot: rank comparison scatter matrix
n_measures = len(MEASURES)
measure_names = list(MEASURES.keys())

fig, axes = plt.subplots(n_measures, n_measures,
                          figsize=(14, 13), sharex=False, sharey=False)

for i, m1 in enumerate(measure_names):
    for j, m2 in enumerate(measure_names):
        ax = axes[i, j]
        if i == j:
            # Diagonal: distribution of scores
            ax.hist(scores_df[m1], bins=12, color=sns.color_palette('husl', 5)[i],
                    edgecolor='white')
            ax.set_title(m1, fontsize=7, fontweight='bold')
        else:
            # Off-diagonal: rank scatter
            r1 = rank_df[m1]
            r2 = rank_df[m2]
            # Colour by rank disagreement
            disagreement = (r1 - r2).abs()
            sc = ax.scatter(r1, r2, c=disagreement, cmap='Reds', s=20, alpha=0.8)

            # Diagonal reference (perfect agreement)
            ax.plot([1, 30], [1, 30], 'k--', alpha=0.3, linewidth=0.8)

            # Spearman rank correlation between measures
            rho, _ = spearmanr(r1, r2)
            ax.set_title(f'ρ={rho:.2f}', fontsize=7)

        if i == n_measures - 1:
            ax.set_xlabel(m2, fontsize=6)
        if j == 0:
            ax.set_ylabel(m1, fontsize=6)
        ax.tick_params(labelsize=5)

plt.suptitle('Feature Ranking Agreement Across Dependence Measures\n'
             '(scatter: rank by measure A vs rank by measure B; ρ = Spearman agreement)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('rank_comparison_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Find features with the largest disagreement across measures
rank_std = rank_df.std(axis=1).sort_values(ascending=False)

print("Features with highest rank disagreement across measures (std of ranks):")
print(rank_std.head(8).round(2))

# For the most disagreed-upon feature, show its scatter vs target under each measure
most_disagreed = rank_std.idxmax()
print(f"\nMost disagreed feature: '{most_disagreed}'")
print(rank_df.loc[most_disagreed])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
feat_vals = X_bc_sc[most_disagreed].values

# Scatter: feature vs target
axes[0].scatter(feat_vals, y_bc + np.random.normal(0, 0.04, len(y_bc)),
                alpha=0.3, s=12, c=y_bc, cmap='RdBu')
axes[0].set_xlabel(most_disagreed)
axes[0].set_ylabel('Class label (jittered)')
axes[0].set_title(f"'{most_disagreed}' vs target")

# Bar: measure scores for this feature (normalised to [0,1] within measure)
raw_scores = {m: scores_df.loc[most_disagreed, m] for m in measure_names}
ranks_feat  = {m: rank_df.loc[most_disagreed, m] for m in measure_names}
colours = sns.color_palette('husl', 5)
bars = axes[1].bar(measure_names, [raw_scores[m] for m in measure_names],
                    color=colours)
axes[1].set_title(f"Scores for '{most_disagreed}'")
axes[1].set_ylabel('Raw score')
for bar, m in zip(bars, measure_names):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                  f'rank {ranks_feat[m]}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('most_disagreed_feature.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Computational Cost Benchmark

dCor and HSIC are $O(N^2)$ — quadratic in sample size. At what $N$ do they become impractical, and how does MI compare?

In [ ]:
sample_sizes = [100, 300, 500, 1000, 2000, 5000]
# Skip HSIC and dCor for N > 3000 to keep this under 2 minutes
SLOW_THRESHOLD = 3000

timing_results = {m: [] for m in MEASURES}

for n_samp in sample_sizes:
    x_bench = np.random.randn(n_samp)
    y_bench = np.random.randn(n_samp)
    y_class = (y_bench > 0).astype(int)

    for meas_name, fn in MEASURES.items():
        # Skip expensive O(N^2) methods at large N
        if meas_name in ('dCor', 'HSIC (RBF)') and n_samp > SLOW_THRESHOLD:
            timing_results[meas_name].append(np.nan)
            continue

        n_reps = max(1, min(10, 5000 // n_samp))  # fewer reps for large N
        t0 = time.perf_counter()
        for _ in range(n_reps):
            if meas_name == 'MI (sklearn)':
                fn(x_bench, y_class)
            else:
                fn(x_bench, y_bench)
        elapsed = (time.perf_counter() - t0) / n_reps
        timing_results[meas_name].append(elapsed)

timing_df = pd.DataFrame(timing_results, index=sample_sizes)
timing_df.index.name = 'N (sample size)'
print("Wall time per call (seconds):")
print(timing_df.round(5).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colours = sns.color_palette('husl', 5)

for ax, log_scale in zip(axes, [False, True]):
    for i, (meas_name, colour) in enumerate(zip(timing_df.columns, colours)):
        vals = timing_df[meas_name].values
        valid_mask = ~np.isnan(vals)
        ax.plot(np.array(sample_sizes)[valid_mask], vals[valid_mask],
                'o-', label=meas_name, color=colour, linewidth=2)

    if log_scale:
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title('Computational Cost (log-log scale)')
    else:
        ax.set_title('Computational Cost (linear scale)')

    ax.set_xlabel('Sample size N')
    ax.set_ylabel('Time per call (seconds)')
    ax.legend(fontsize=8)

plt.suptitle('Computational Cost: Linear vs O(N²) Methods',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('computational_cost_benchmark.png', dpi=120, bbox_inches='tight')
plt.show()

# At what N does dCor/HSIC take > 1 second?
for meas in ['dCor', 'HSIC (RBF)']:
    vals = timing_df[meas].dropna()
    if len(vals) > 0:
        last_n = vals.index[-1]
        last_t = vals.iloc[-1]
        # Extrapolate quadratic scaling
        n_1sec_est = int(last_n * (1.0 / last_t)**0.5)
        print(f"{meas}: at N={last_n} takes {last_t:.3f}s → estimated N for 1s: ~{n_1sec_est:,}")

## 5. Practical Recommendation Framework

Summarise when to use each measure in a decision table.

In [ ]:
# Measure recommendation table
recommendations = pd.DataFrame([
    {
        'Measure': 'Pearson |r|',
        'Use when': 'Quick sanity check; baseline',
        'Detects': 'Linear only',
        'N requirement': 'Any',
        'Cost': 'O(N)',
        'Limitation': 'Misses nonlinear, monotone relationships',
    },
    {
        'Measure': 'Spearman |ρ|',
        'Use when': 'Monotone relationships; ordinal data',
        'Detects': 'Monotone',
        'N requirement': '> 30',
        'Cost': 'O(N log N)',
        'Limitation': 'Misses non-monotone (e.g. sinusoidal, XOR)',
    },
    {
        'Measure': 'MI (sklearn)',
        'Use when': 'Standard choice for continuous 1-D features',
        'Detects': 'Arbitrary (all types)',
        'N requirement': '> 100 (reliable >500)',
        'Cost': 'O(N log N)',
        'Limitation': 'Requires binning for discrete; small-sample bias',
    },
    {
        'Measure': 'dCor',
        'Use when': 'Multivariate features; robust to outliers',
        'Detects': 'Arbitrary; multivariate X, Y',
        'N requirement': '> 30 (bias-corrected)',
        'Cost': 'O(N²)',
        'Limitation': 'Expensive; insensitive in very high dimensions',
    },
    {
        'Measure': 'HSIC (RBF)',
        'Use when': 'Flexible kernel; need formal independence test',
        'Detects': 'Arbitrary; kernel-tunable sensitivity',
        'N requirement': '> 50',
        'Cost': 'O(N²)',
        'Limitation': 'Bandwidth-sensitive; unbounded scale',
    },
])

print("Practical recommendation framework:")
for _, row in recommendations.iterrows():
    print(f"\n{row['Measure']}")
    print(f"  Use when:    {row['Use when']}")
    print(f"  Detects:     {row['Detects']}")
    print(f"  N required:  {row['N requirement']}")
    print(f"  Cost:        {row['Cost']}")
    print(f"  Limitation:  {row['Limitation']}")

## 6. Exercise: Consensus Ranking

**Task:** Build a consensus ranking that combines all five measures. The consensus rank for feature $f$ is the average of its ranks under all five measures. Features that rank highly across all measures are the most robustly relevant.

**Steps:**
1. Compute per-measure ranks from `rank_df` (already computed above)
2. Average the ranks across measures
3. Sort by mean rank (ascending — rank 1 is best)
4. Identify features that rank in the top 5 by at least 3 measures

**Extension:** Add uncertainty to the consensus by computing the standard deviation of ranks. A feature with low mean rank AND low rank standard deviation is more reliably relevant.

In [ ]:
# YOUR CODE HERE
# ---------------

# consensus_ranks = ...
# consensus_std   = ...
# consensus_df    = pd.DataFrame({'mean_rank': ..., 'std_rank': ...}).sort_values('mean_rank')

# top_consensus = consensus_df.head(10)
# print(top_consensus)

In [ ]:
# Reference solution
consensus_mean = rank_df.mean(axis=1)
consensus_std  = rank_df.std(axis=1)

consensus_df = pd.DataFrame({
    'consensus_rank_mean': consensus_mean,
    'consensus_rank_std':  consensus_std,
}).sort_values('consensus_rank_mean')

# Features in top 5 by at least 3 measures
top5_per_measure = set()
for col in rank_df.columns:
    top5_per_measure |= set(rank_df[col].nsmallest(5).index)

count_top5 = {feat: (rank_df.loc[feat] <= 5).sum() for feat in rank_df.index}
robust_features = [feat for feat, cnt in count_top5.items() if cnt >= 3]

print("Consensus feature ranking (mean of per-measure ranks):")
print(consensus_df.head(10).round(2))
print(f"\nRobust features (top-5 in >= 3 measures): {robust_features}")

# Visualise consensus
fig, ax = plt.subplots(figsize=(11, 6))
top10_feats = consensus_df.head(10).index
y_pos = np.arange(len(top10_feats))

ax.barh(y_pos, consensus_df.loc[top10_feats, 'consensus_rank_mean'],
         xerr=consensus_df.loc[top10_feats, 'consensus_rank_std'],
         align='center', color='#3498db', alpha=0.8, capsize=4,
         error_kw={'linewidth': 1.5})
ax.set_yticks(y_pos)
ax.set_yticklabels(top10_feats, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Mean rank across all 5 measures (lower = more relevant)')
ax.set_title('Consensus Feature Ranking (Top 10)\nError bars = std across measures',
              fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('consensus_ranking.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Summary

### Key Takeaways

1. **No single measure wins on all relationship types.** Pearson and Spearman miss nonlinear patterns. MI, dCor, and HSIC all detect arbitrary dependencies, but with different sensitivities and computational costs.

2. **For 1-D continuous features, MI (sklearn) is the default choice.** It is $O(N \log N)$, handles any dependency shape, and is the fastest nonlinear measure.

3. **dCor and HSIC add value for multivariate features and formal independence tests.** Their $O(N^2)$ cost is manageable for $N < 5000$. For larger datasets, use random Fourier feature approximations.

4. **Consensus ranking is robust.** Features that rank highly across multiple measures are the safest choices. Large rank disagreement indicates unusual dependency structure — worth investigating.

### What's Next

Module 1 exercises — filter_methods_exercises.py: implement distance correlation and Relief-based selectors from scratch.

### Additional Resources
- Székely, G.J. et al. (2007). *Measuring and testing dependence by correlation of distances.* Annals of Statistics.
- Gretton, A. et al. (2012). *A kernel two-sample test.* JMLR.